# Softmax

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

把一组任意实数转换成概率分布：
- 每个输出值在 $(0, 1)$ 之间
- 所有输出值之和为 1
- 保留大小关系：较大的输入对应较大的概率

## 数值稳定性问题

直接计算 $e^{x_i}$ 当 $x_i$ 很大时会溢出（`inf`），所以实现时需要减去最大值：

$$\text{softmax}(x_i) = \frac{e^{x_i - x_{\max}}}{\sum_j e^{x_j - x_{\max}}}$$

数学上完全等价（分子分母同除 $e^{x_{\max}}$），但避免了数值溢出。

In [1]:
import torch

In [2]:
def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    x_max = x.max(dim=dim, keepdim=True).values  # 减最大值，防止 exp 溢出
    e_x = torch.exp(x - x_max)
    return e_x / e_x.sum(dim=dim, keepdim=True)

## 验证

In [3]:
x = torch.tensor([1.0, 2.0, 3.0])
print("Output:", my_softmax(x, dim=-1))
print("Sum:   ", my_softmax(x, dim=-1).sum())   # 应为 1.0
print("Ref:   ", torch.softmax(x, dim=-1))       # 应与 Output 一致

Output: tensor([0.0900, 0.2447, 0.6652])
Sum:    tensor(1.)
Ref:    tensor([0.0900, 0.2447, 0.6652])


## keepdim 的作用

`keepdim=True` 让结果保留被 reduce 的那个维度（大小变为 1），这样才能和原 tensor 做广播除法。

```python
x.shape        # (3,)
x.max(dim=-1, keepdim=True).values.shape   # (1,)  ← keepdim=True
x.max(dim=-1, keepdim=False).values.shape  # ()    ← 维度消失，无法广播
```

对于 2D 以上的 tensor（如 attention scores `(B, T, T)`），keepdim 尤为关键。